In [1]:
import numpy as np
from io import StringIO
import platform
import sys
from pathlib import Path

project_root = next(
    (candidate for parent in (Path.cwd(), *Path.cwd().parents)
     for candidate in (parent, parent / "NumCompute")
     if (candidate / "numcompute" / "__init__.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open this notebook from the NumCompute project folder.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from numcompute.io import read_csv as load_csv
from numcompute.preprocessing import SimpleImputer as Imputer, StandardScaler, OneHotEncoder
from numcompute.sort_search import topk, quickselect, binary_search
from numcompute.rank import compute_rank as rank, percentile
from numcompute.stats import compute_basic_stats, histogram, quantiles
from numcompute.metrics import accuracy, precision_recall_f1, mse
from numcompute.optim import grad, jacobian
from numcompute.pipeline import Pipeline
from numcompute.benchmarking import compare_sum, python_sum, numpy_sum

In [2]:
csv_data = StringIO("feature_1,feature_2\n1.0,\n2.0,5.0\n3.0,6.0\n")
X = load_csv(csv_data, delimiter=",", dtype=float, skip_header=1)
print("CSV data:")
print(X)
print("Shape:", X.shape)
print("Missing values:", np.isnan(X).sum())

CSV data:
[[ 1. nan]
 [ 2.  5.]
 [ 3.  6.]]
Shape: (3, 2)
Missing values: 1


In [3]:
pipe = Pipeline([
    ('imputer', Imputer(fill_value=0.0)),
    ('scaler', StandardScaler())
])
X_tr = pipe.fit_transform(X)
X_tr

array([[-1.22474487, -1.3970014 ],
       [ 0.        ,  0.50800051],
       [ 1.22474487,  0.88900089]])

In [4]:
cats = np.array(['red', 'blue', 'red'])
enc = OneHotEncoder()
enc.fit_transform(cats.reshape(-1, 1))

array([[0., 1.],
       [1., 0.],
       [0., 1.]])

In [5]:
values = np.array([10, 4, 8, 2, 9])
print("Top-k values and indices:", topk(values, 3))
print("Third smallest value:", quickselect(values, 3))
print("Binary search:", binary_search(np.array([1, 3, 5, 7]), 5))
print("Average ranks:", rank(np.array([10, 20, 20, 30]), method='average'))
print("50th percentile:", percentile(values, 50))

Top-k values and indices: (array([10,  9,  8]), array([0, 4, 2]))
Third smallest value: 8
Binary search: (np.int64(2), True)
Average ranks: [1.  2.5 2.5 4. ]
50th percentile: 8.0


In [6]:
x = np.array([1, 2, 3, 4, 5], dtype=float)
compute_basic_stats(x)["mean"], compute_basic_stats(x)["std"], histogram(x, bins=2), quantiles(x, [0.25, 0.5, 0.75])

(np.float64(3.0),
 np.float64(1.4142135623730951),
 (array([2, 3]), array([1., 3., 5.])),
 array([2., 3., 4.]))

In [7]:
y_true = np.array([1, 0, 1, 1])
y_pred = np.array([1, 0, 0, 1])
precision_score, recall_score, f1_score = precision_recall_f1(y_true, y_pred)
print("Accuracy:", accuracy(y_true, y_pred))
print("Precision (macro):", precision_score)
print("Recall (macro):", recall_score)
print("F1 (macro):", f1_score)
print("MSE:", mse(np.array([1.0, 2.0, 3.0]), np.array([1.0, 2.0, 4.0])))

Accuracy: 0.75
Precision (macro): 0.75
Recall (macro): 0.8333333333333333
F1 (macro): 0.7333333333333334
MSE: 0.3333333333333333


In [8]:
def f(v):
    return v[0] ** 2 + v[1] ** 2

def F(v):
    return np.array([v[0] + v[1], v[0] * v[1]])

grad(f, np.array([3.0, 4.0])), jacobian(F, np.array([2.0, 3.0]))

(array([6., 8.]),
 array([[1., 1.],
        [3., 2.]]))

In [9]:
n, repeat, seed = 100000, 5, 42
benchmark_input = np.random.default_rng(seed).random(n)
np.testing.assert_allclose(python_sum(benchmark_input), numpy_sum(benchmark_input))
results = compare_sum(n=n, repeat=repeat, seed=seed)
np.testing.assert_allclose(results["python_sum"]["result"], results["numpy_sum"]["result"])
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Platform:", platform.platform())
print(f"Input size: {n}; repeats: {repeat}; seed: {seed}")
print(f"{'Implementation':<18} {'Mean (ms)':>12} {'Min (ms)':>12} {'Max (ms)':>12}")
for name, timing in results.items():
    print(f"{name:<18} {timing['mean'] * 1000:>12.4f} {timing['min'] * 1000:>12.4f} {timing['max'] * 1000:>12.4f}")
speedup = results["python_sum"]["mean"] / results["numpy_sum"]["mean"]
print(f"Speedup (mean time): {speedup:.2f}x")

Python: 3.10.11
NumPy: 2.2.6
Platform: Windows-10-10.0.26200-SP0
Input size: 100000; repeats: 5; seed: 42
Implementation        Mean (ms)     Min (ms)     Max (ms)
python_sum               4.5391       4.3955       4.6652
numpy_sum                0.0162       0.0157       0.0175
Speedup (mean time): 279.50x
